# PyTorch 学习

## 0. Conda 环境指令

### 0.1 列出所有环境的指令

```bash
conda env list
```

### 0.2 创建环境

```bash
conda create -n dl python=3.12
```

### 0.3 激活环境

```bash
conda activate dl
```

### 0.4 退出环境

```bash
conda deactivate
```

### 0.5 删除环境

```bash
conda remove -n dl --all
```

### 0.6 列出环境内包

```bash
conda list -n dl
```

In [2]:
import torch

# 打印 PyTorch 版本，确认当前环境装的是哪个版本。
print(torch.__version__)
# 查看当前机器是否能使用 CUDA，也就是 NVIDIA GPU。
print(torch.cuda.is_available())

# 统一用 device 管理模型和数据。
# 如果有 GPU，就把张量和模型放到 cuda；否则就退回 cpu。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# 只有在 CUDA 可用时，才去读取 GPU 名称，避免报错。
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


2.11.0+cu128
True
device: cuda
NVIDIA GeForce RTX 5070 Ti


---

## 1. PyTorch 张量和设备 Tensor and Device

### 1.1 定义

`Tensor` 是 PyTorch 中最基础的数据结构，可以理解为支持 GPU 加速和自动求导的多维数组。神经网络里的输入、标签、权重、梯度、损失值，最后都会以 tensor 的形式参与计算。

常见属性：

| 属性 | 含义 | 例子 |
| --- | --- | --- |
| `shape` / `size()` | 张量形状 | `(batch_size, feature_dim)` |
| `dtype` | 数据类型 | `torch.float32`, `torch.long` |
| `device` | 数据所在设备 | `cpu`, `cuda:0` |
| `requires_grad` | 是否记录梯度 | 模型参数一般为 `True` |

### 1.2 常用创建函数

| 函数 | 作用 | 常用写法 |
| --- | --- | --- |
| `torch.tensor(data)` | 从 Python 列表或数组创建 tensor | `torch.tensor([1, 2, 3])` |
| `torch.zeros(shape)` | 创建全 0 tensor | `torch.zeros(3, 4)` |
| `torch.ones(shape)` | 创建全 1 tensor | `torch.ones(3, 4)` |
| `torch.randn(shape)` | 标准正态随机数 | `torch.randn(32, 10)` |
| `torch.randint(low, high, shape)` | 整数随机标签 | `torch.randint(0, 3, (32,))` |
| `torch.arange(start, end)` | 等差整数序列 | `torch.arange(0, 10)` |
| `torch.linspace(start, end, steps)` | 等间隔浮点序列 | `torch.linspace(0, 1, 5)` |

### 1.3 常用形状操作

| 函数 | 作用 | 注意点 |
| --- | --- | --- |
| `x.view(...)` | 改变形状 | 要求内存连续，不连续时先用 `contiguous()` |
| `x.reshape(...)` | 改变形状 | 比 `view` 更灵活，常用 |
| `x.unsqueeze(dim)` | 增加一个维度 | 常用于补 batch/channel 维度 |
| `x.squeeze(dim)` | 删除长度为 1 的维度 | 不指定 `dim` 时可能删多了 |
| `x.permute(...)` | 调整维度顺序 | 图像常见 `NCHW` / `NHWC` 转换 |
| `torch.cat(list, dim)` | 在已有维度拼接 | 除拼接维度外，其他维度必须相同 |
| `torch.stack(list, dim)` | 新增一个维度后堆叠 | 输入 tensor 形状必须完全相同 |

### 1.4 设备和类型转换

- `x.to(device)`：把数据移动到 CPU 或 GPU。
- `x.float()`：转成浮点数，模型输入通常需要这个类型。
- `y.long()`：转成长整型，`nn.CrossEntropyLoss` 的分类标签通常需要这个类型。
- `x.detach()`：从计算图中分离，常用于记录中间结果。
- `x.cpu().numpy()`：把 tensor 转成 NumPy 数组，GPU tensor 需要先 `.cpu()`。

### 1.5 注意点

- 模型和数据必须在同一个设备上，否则会报 device mismatch。
- 分类标签一般是一维 `LongTensor`，形状是 `[batch_size]`，不是 one-hot。
- 神经网络输入通常是 `float32`，不要把输入误传成 `long`。

In [3]:
import torch

# 固定随机种子，方便每次运行都得到相同结果，便于调试和复现。
torch.manual_seed(42)

# 造一批演示数据：
# x 是输入特征，32 表示样本数，20 表示每个样本有 20 个特征。
x = torch.randn(32, 20)
# y 是分类标签，形状为 [32]，每个值只能是 0 或 1。
y = torch.randint(0, 2, (32,))

# 把数据搬到 device 上，保证后面模型和数据在同一个设备里。
x = x.to(device)
y = y.to(device)

print("x shape:", x.shape)
print("x dtype:", x.dtype)
print("x device:", x.device)
print("y shape:", y.shape)
print("y dtype:", y.dtype)


x shape: torch.Size([32, 20])
x dtype: torch.float32
x device: cuda:0
y shape: torch.Size([32])
y dtype: torch.int64


---

## 2. 数据集和 DataLoader Dataset and DataLoader

### 2.1 定义

训练模型时，数据通常不会一次性全部送进模型，而是按 batch 分批读取。PyTorch 里常用 `Dataset` 表示“数据如何取”，用 `DataLoader` 表示“数据如何分批、打乱、并行加载”。

### 2.2 常用函数和类

| 函数 / 类 | 作用 | 常用参数 |
| --- | --- | --- |
| `TensorDataset(x, y)` | 把已有 tensor 包成数据集 | `x`, `y` 第一维长度要一致 |
| `Dataset` | 自定义数据集基类 | 实现 `__len__` 和 `__getitem__` |
| `DataLoader(dataset)` | 分批加载数据 | `batch_size`, `shuffle`, `num_workers` |
| `random_split(dataset, lengths)` | 随机划分训练集和验证集 | 可传 `generator` 固定随机种子 |
| `transforms.Compose([...])` | 按顺序组合多个图像预处理步骤 | 常用于 `torchvision` 图像数据集 |

### 2.3 DataLoader 常用参数

| 参数 | 作用 | 常见选择 |
| --- | --- | --- |
| `batch_size` | 每次送进模型的样本数 | 32、64、128 |
| `shuffle` | 每个 epoch 是否打乱 | 训练集 `True`，验证集 `False` |
| `num_workers` | 数据加载进程数 | Windows/Jupyter 中可以先用 0 |
| `drop_last` | 是否丢掉最后不足一个 batch 的数据 | BatchNorm 训练时可考虑 `True` |
| `pin_memory` | 加快 CPU 到 GPU 拷贝 | 使用 CUDA 时常设为 `True` |

### 2.4 torchvision.transforms 的用法

`transforms` 用来把原始图片处理成模型能直接接收的张量。它通常放在数据集的 `transform` 参数里，Dataset 每次取出图片时会自动执行这些处理。

| 写法 | 作用 | 注意点 |
| --- | --- | --- |
| `transforms.Compose([...])` | 把多个处理步骤串起来 | 按列表顺序依次执行 |
| `transforms.Resize(224)` | 调整图片尺寸 | 单个整数会保持宽高比例；要固定高宽可写 `(224, 224)` |
| `transforms.ToTensor()` | 把 PIL 图片或 NumPy 数组转成 tensor | 形状从 `HWC` 变成 `CHW`，像素从 `[0, 255]` 变成 `[0, 1]` |
| `transforms.Normalize(mean, std)` | 按均值和标准差标准化 | 灰度图写一个值，如 `(0.5,)`；RGB 图写三个值 |

常见写法：

```python
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
```

其中 `Resize` 决定图片输入大小，`ToTensor` 决定数据格式。对 CNN 来说，最终输入一般是 `[C, H, W]`，例如 FashionMNIST 灰度图处理后是 `[1, 224, 224]`。

### 2.5 注意点

- `Dataset.__getitem__` 返回的通常是 `(input, label)`。
- 训练集要打乱，验证集和测试集一般不打乱。
- 数据增强通常写在 `Dataset` 或 transform 里，不写在训练循环里。
- `Compose` 里的顺序很重要，一般先 resize / crop，再 `ToTensor`，最后 normalize。
- `FashionMNIST` 是灰度图，通道数是 1；普通彩色图片通常是 3 通道。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

# 这里先造一个小型二分类数据集，方便演示 DataLoader 的用法。
# features: 输入特征，shape = [1000, 20]
features = torch.randn(1000, 20)
# labels: 每条样本对应一个类别标签，0 或 1
labels = torch.randint(0, 2, (1000,))

# TensorDataset 会把 features 和 labels 一一配对，
# 之后通过索引就能同时拿到一条样本和它的标签。
dataset = TensorDataset(features, labels)

# 按 8:2 划分训练集和验证集。
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size

# random_split 会随机切分数据集。
# 这里手动固定随机种子，避免每次切分结果不一样。
train_set, val_set = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

# DataLoader 负责按 batch 取数据。
# 训练集打乱顺序，能减少模型记住数据顺序的影响。
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
# 验证集不需要打乱，直接顺序读取即可。
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)

# 取出一个 batch 看看形状，确认 DataLoader 正常工作。
batch_x, batch_y = next(iter(train_loader))
print(batch_x.shape, batch_y.shape)


tensor([[-1.9553,  1.9156,  0.7092,  ..., -3.2090,  0.1218,  0.2822],
        [-1.1792, -1.0074,  0.6639,  ..., -2.4527, -0.9881,  0.0223],
        [-1.3499, -0.3060, -0.4069,  ...,  0.0260,  0.5544, -1.1809],
        ...,
        [ 1.1668,  1.5771,  0.5159,  ...,  2.0694, -0.2438, -0.7609],
        [ 1.9260,  0.1984, -1.8696,  ...,  1.1996, -2.7301, -0.2186],
        [ 0.7988, -0.8710, -0.0710,  ...,  0.5766,  1.2968,  0.8607]])
torch.Size([64, 20]) torch.Size([64])


In [ ]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    """
    自定义数据集。
    继承 Dataset 之后，只要实现 __len__ 和 __getitem__，
    就可以直接配合 DataLoader 使用。
    """
    def __init__(self, x, y):
        # 特征转成 float，方便后面的线性层/卷积层计算。
        self.x = x.float()
        # 标签转成 long，因为 CrossEntropyLoss 需要类别索引类型。
        self.y = y.long()

    def __len__(self):
        # 返回数据集里一共有多少条样本。
        return len(self.x)
    
    def __getitem__(self, index):
        # 根据索引返回一条样本和它的标签。
        return self.x[index], self.y[index]

# 用自定义 Dataset 封装现成张量。
custom_dataset = MyDataset(features, labels)
print(len(custom_dataset))
print(custom_dataset[0][0].shape, custom_dataset[0][1])


1000
torch.Size([20]) tensor(0)


In [ ]:
from torchvision import transforms
from torchvision.datasets import FashionMNIST

# Compose 会按顺序执行里面的每一步处理。
# FashionMNIST 原图是 28x28 的灰度图，这里把它放大到 224x224，方便接到一些 CNN 模型里。
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# transform 会在 Dataset 取样本时自动执行。
train_dataset = FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

image, label = train_dataset[0]
print(image.shape, label)  # torch.Size([1, 224, 224]), 标签是 0-9 的类别编号


---

## 3. 搭建神经网络 nn.Module

### 3.1 定义

PyTorch 中搭建神经网络最常用的方法是继承 `nn.Module`：

1. 在 `__init__` 中定义需要学习参数的层，比如 `nn.Linear`、`nn.Conv2d`。
2. 在 `forward` 中写清楚数据从输入到输出的计算过程。
3. 实例化模型后，用 `.to(device)` 把模型移动到 CPU 或 GPU。

### 3.2 常用层

| 层 | 作用 | 常用场景 |
| --- | --- | --- |
| `nn.Linear(in, out)` | 全连接层 | 表格数据、MLP、分类头 |
| `nn.Conv2d(in_ch, out_ch, kernel_size)` | 二维卷积 | 图像特征提取 |
| `nn.MaxPool2d(kernel_size)` | 最大池化 | 降低图像尺寸 |
| `nn.BatchNorm1d/2d(num_features)` | 批归一化 | 稳定训练、加快收敛 |
| `nn.Dropout(p)` | 随机丢弃神经元 | 减少过拟合 |
| `nn.Flatten()` | 展平特征 | CNN 接全连接层前常用 |
| `nn.Embedding(num_embeddings, dim)` | 词表索引转向量 | NLP 输入表示 |
| `nn.LSTM(...)` | 循环神经网络 | 序列数据 |
| `nn.TransformerEncoderLayer(...)` | Transformer 编码层 | 文本、序列、视觉 Transformer |

### 3.3 常用激活函数

| 激活函数 | 作用 | 注意点 |
| --- | --- | --- |
| `nn.ReLU()` | 保留正数，负数变 0 | 最常用，简单稳定 |
| `nn.LeakyReLU()` | 负数保留小斜率 | 可缓解 ReLU 死亡问题 |
| `nn.Sigmoid()` | 输出压到 0 到 1 | 二分类概率或门控结构 |
| `nn.Tanh()` | 输出压到 -1 到 1 | RNN 中较常见 |
| `nn.GELU()` | 平滑非线性 | Transformer 中常见 |
| `nn.Softmax(dim)` | 转成概率分布 | 训练 `CrossEntropyLoss` 前不要手动 softmax |

### 3.4 `nn.Module` 和 `nn.Sequential` 的选择

- 网络结构简单、按顺序堆叠时，可以用 `nn.Sequential`。
- 需要分支、跳连、多个输入输出或复杂逻辑时，用自定义 `nn.Module`。
- `forward` 里可以写普通 Python 控制流，但要保证输入输出形状清楚。

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, output_dim=2):
        super().__init__()

        # nn.Sequential 按顺序堆叠层，写起来更紧凑。
        self.net = nn.Sequential(
            # 第一层：把 20 维输入映射到 64 维隐藏特征。
            nn.Linear(input_dim, hidden_dim),
            # BatchNorm 用来稳定数值分布，让训练更平稳。
            nn.BatchNorm1d(hidden_dim),
            # ReLU 引入非线性，否则多层线性叠加还是线性模型。
            nn.ReLU(),
            # Dropout 在训练时随机丢弃一部分神经元，减少过拟合。
            nn.Dropout(p=0.2),
            # 再堆一层隐藏层，让模型学到更复杂的特征组合。
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            # 最后一层输出 2 个值，对应二分类的两个类别得分。
            # 这里输出的是 logits，不是概率。
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        # 前向传播直接把输入送进顺序网络。
        return self.net(x)

# 把模型放到 device 上，和数据保持一致。
model = MLP(input_dim=20, hidden_dim=64, output_dim=2).to(device)
print(model)

# 统计所有需要训练的参数数量，方便确认模型规模。
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params:", trainable_params)


In [ ]:
# 前向传播示例：输入一个 batch，查看模型输出的形状。
batch_x, batch_y = next(iter(train_loader))
batch_x = batch_x.to(device)

# 模型输出的是每个类别的原始分数 logits。
logits = model(batch_x)
# logits 的形状是 [batch_size, num_classes]。
print("logits shape:", logits.shape)
# 用 argmax 取分数最高的类别，得到预测标签。
print("pred shape:", logits.argmax(dim=1).shape)


---

## 4. 损失函数、优化器和学习率 Loss / Optimizer / Scheduler

### 4.1 损失函数

损失函数用来衡量模型预测和真实标签之间的差距。训练的目标就是让损失越来越小。

| 损失函数 | 作用 | 输入要求 |
| --- | --- | --- |
| `nn.MSELoss()` | 均方误差 | 回归任务常用 |
| `nn.L1Loss()` | 平均绝对误差 | 回归任务，对异常值相对不敏感 |
| `nn.CrossEntropyLoss()` | 多分类交叉熵 | 输入 raw logits，标签是类别编号 |
| `nn.BCEWithLogitsLoss()` | 二分类或多标签分类 | 输入 raw logits，内部包含 sigmoid |
| `nn.NLLLoss()` | 负对数似然 | 通常配合 `LogSoftmax` |

`CrossEntropyLoss` 的常见输入：

- `logits`：形状 `[batch_size, num_classes]`，不需要先 `softmax`。
- `target`：形状 `[batch_size]`，类型为 `torch.long`，值是类别编号。

### 4.2 优化器

| 优化器 | 特点 | 常用场景 |
| --- | --- | --- |
| `torch.optim.SGD` | 简单，可配合 momentum | 传统 CNN、需要细调时 |
| `torch.optim.Adam` | 自适应学习率，收敛快 | 通用默认选择 |
| `torch.optim.AdamW` | Adam 加改进版权重衰减 | Transformer、现代深度学习常用 |

常用参数：

- `lr`：学习率，太大容易震荡，太小训练很慢。
- `weight_decay`：权重衰减，用于正则化，减少过拟合。
- `momentum`：SGD 常用，加快收敛。

### 4.3 学习率调度器

| 调度器 | 作用 | 使用方式 |
| --- | --- | --- |
| `StepLR` | 每隔固定 epoch 衰减学习率 | `scheduler.step()` |
| `CosineAnnealingLR` | 按余弦曲线下降 | 常用于较长训练 |
| `ReduceLROnPlateau` | 指标不提升时降低学习率 | `scheduler.step(val_loss)` |

### 4.4 标准更新顺序

1. `optimizer.zero_grad()`：清空上一轮梯度。
2. `logits = model(x)`：前向传播。
3. `loss = criterion(logits, y)`：计算损失。
4. `loss.backward()`：反向传播，计算梯度。
5. `optimizer.step()`：更新模型参数。
6. `scheduler.step()`：按需要更新学习率。

In [ ]:
# CrossEntropyLoss 用于多分类/二分类的类别索引任务。
# 它要求模型输出 logits，标签是类别编号（long 类型）。
criterion = nn.CrossEntropyLoss()

# AdamW 是 Adam 的变体，常用于深度学习训练。
# lr 控制每次更新的步长，weight_decay 是权重衰减，用来抑制过拟合。
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

# StepLR 每隔 step_size 个 epoch 把学习率乘上 gamma。
# 学习率后期变小，通常能让训练更稳定。
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5,
)

print(criterion)
print(optimizer)


---

## 5. 训练模型的标准流程 Training Loop

### 5.1 训练循环的核心概念

训练循环主要做三件事：

1. 从 `DataLoader` 中取一个 batch。
2. 让模型预测，计算损失，再反向传播更新参数。
3. 记录 loss、accuracy 等指标，用来判断模型是否真的在学习。

### 5.2 `model.train()` 和 `model.eval()`

| 模式 | 作用 | 影响的层 |
| --- | --- | --- |
| `model.train()` | 训练模式 | `Dropout` 会随机丢弃，`BatchNorm` 会更新统计量 |
| `model.eval()` | 验证/推理模式 | `Dropout` 关闭，`BatchNorm` 使用已有统计量 |

验证和推理时还要配合：

```python
with torch.no_grad():
    ...
```

这样不会记录梯度，能减少显存占用并加快计算。

### 5.3 常见指标

分类任务最常见的是准确率 Accuracy：

$$
Accuracy = \frac{预测正确的样本数}{总样本数}
$$

训练时不能只看训练集指标，还要看验证集指标：

- 训练集 loss 降低，验证集 loss 也降低：通常说明训练有效。
- 训练集 loss 降低，验证集 loss 升高：可能过拟合。
- 两边 loss 都很高：可能模型太弱、学习率不合适或数据特征不足。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # 训练模式：启用 Dropout，BatchNorm 使用当前 batch 的统计量。
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_x, batch_y in loader:
        # 把一个 batch 的数据搬到同一设备上。
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # 清空上一次迭代留下的梯度。
        optimizer.zero_grad(set_to_none=True)
        # 前向传播：得到每个类别的 logits。
        logits = model(batch_x)
        # 计算预测和真实标签之间的损失。
        loss = criterion(logits, batch_y)
        # 反向传播：根据 loss 计算每个参数的梯度。
        loss.backward()

        # 梯度裁剪可以避免梯度突然过大，复杂模型中更常见。
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 根据梯度更新模型参数。
        optimizer.step()

        # 当前 batch 的样本数。
        batch_size = batch_x.size(0)
        # loss.item() 是当前 batch 的平均损失，乘 batch_size 后可累计总损失。
        total_loss += loss.item() * batch_size
        # argmax 取预测类别，和真实标签比较后统计正确个数。
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total_samples += batch_size

    # 用总损失除以总样本数，得到整轮训练的平均损失。
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    # 评估模式：关闭 Dropout，BatchNorm 使用训练期间累计的统计量。
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # 评估时只做前向传播，不需要反向传播，也不会保存梯度。
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        batch_size = batch_x.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


In [ ]:
num_epochs = 5

# 训练若干轮，每一轮都先训练再验证。
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )
    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
        device,
    )

    # 每个 epoch 结束后更新一次学习率。
    scheduler.step()

    # 打印当前 epoch 的训练和验证指标，方便观察是否收敛。
    print(
        f"epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )


---

## 6. 验证、推理和概率输出 Inference

### 6.1 推理流程

推理时不更新参数，只需要前向传播：

1. `model.eval()`：切换到推理模式。
2. `with torch.no_grad()`：关闭梯度记录。
3. `logits = model(x)`：得到模型原始输出。
4. `pred = logits.argmax(dim=1)`：得到预测类别。
5. 如果需要概率，再使用 `torch.softmax(logits, dim=1)`。

### 6.2 logits、概率和类别

| 名称 | 含义 | 常用代码 |
| --- | --- | --- |
| logits | 模型最后一层的原始输出 | `logits = model(x)` |
| probability | 每个类别的概率 | `prob = torch.softmax(logits, dim=1)` |
| prediction | 概率最大的类别编号 | `pred = logits.argmax(dim=1)` |

训练 `CrossEntropyLoss` 时用 logits；展示结果或解释预测时才需要概率。

In [ ]:
# 切到评估模式，保证 Dropout/BatchNorm 的行为稳定。
model.eval()

# torch.no_grad() 表示下面的推理过程不记录梯度，省内存也更快。
with torch.no_grad():
    # 从验证集里取一个 batch 做演示。
    sample_x, sample_y = next(iter(val_loader))
    sample_x = sample_x.to(device)

    # 先得到 logits，再转换成概率和类别预测。
    logits = model(sample_x)
    probabilities = torch.softmax(logits, dim=1)
    predictions = logits.argmax(dim=1)

print("probabilities shape:", probabilities.shape)
print("first 5 predictions:", predictions[:5].cpu().tolist())


---

## 7. 保存和加载模型 Save and Load

### 7.1 推荐方式

PyTorch 推荐保存 `state_dict`，也就是模型参数字典，而不是直接保存整个模型对象。

| 目标 | 常用代码 | 说明 |
| --- | --- | --- |
| 保存模型参数 | `torch.save(model.state_dict(), path)` | 只保存权重 |
| 加载模型参数 | `model.load_state_dict(torch.load(path))` | 需要先创建同结构模型 |
| 保存训练断点 | 保存模型、优化器、epoch、loss | 方便继续训练 |
| 跨设备加载 | `map_location=device` | GPU/CPU 切换时常用 |

### 7.2 注意点

- 加载 `state_dict` 前，模型结构必须和保存时一致。
- 继续训练时，还要加载 optimizer 和 scheduler 的状态。
- 推理加载后要调用 `model.eval()`。
- 文件后缀常用 `.pt` 或 `.pth`。

In [ ]:
checkpoint_path = "mlp_checkpoint.pt"

# 保存 checkpoint 不只保存模型参数，还保存优化器和学习率调度器状态，
# 这样下次可以从中断位置继续训练，而不是只拿到一个模型权重文件。
torch.save(
    {
        "epoch": num_epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    },
    checkpoint_path,
)

# 重新创建一个结构完全一样的模型和训练组件，用来演示“加载再恢复”。
loaded_model = MLP(input_dim=20, hidden_dim=64, output_dim=2).to(device)
loaded_optimizer = torch.optim.AdamW(loaded_model.parameters(), lr=1e-3, weight_decay=1e-4)
loaded_scheduler = torch.optim.lr_scheduler.StepLR(loaded_optimizer, step_size=5, gamma=0.5)

# map_location=device 可以确保无论原来保存在哪个设备上，都能加载到当前设备。
checkpoint = torch.load(checkpoint_path, map_location=device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
loaded_scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
loaded_model.eval()

print("loaded epoch:", checkpoint["epoch"])


---

## 8. CNN 图像分类模板 Convolutional Neural Network

### 8.1 输入形状

PyTorch 的二维卷积默认输入形状是：

$$
[batch\_size, channels, height, width]
$$

例如灰度手写数字图像可以写成 `[N, 1, 28, 28]`，RGB 图片可以写成 `[N, 3, H, W]`。

### 8.2 常见组合

CNN 中常见结构是：

1. `Conv2d` 提取局部特征。
2. `ReLU` 增加非线性。
3. `MaxPool2d` 降低空间尺寸。
4. `Flatten` 展平成向量。
5. `Linear` 输出类别 logits。

### 8.3 注意点

- 输入图片维度不要写成 `[N, H, W, C]`，这通常是 NumPy/OpenCV 风格。
- 接全连接层前要算清楚 flatten 后的特征维度。
- 如果维度不确定，可以先用一个 dummy tensor 跑一遍看输出形状。

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # 特征提取部分：卷积 + 归一化 + 激活 + 池化。
        self.features = nn.Sequential(
            # 输入是 1 通道灰度图，输出 16 个卷积特征图。
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            # 2x2 池化把 28x28 缩小到 14x14。
            nn.MaxPool2d(kernel_size=2),  # 28x28 -> 14x14
            # 第二个卷积块，把通道数提到 32。
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 再次池化，把 14x14 缩小到 7x7。
            nn.MaxPool2d(kernel_size=2),  # 14x14 -> 7x7
        )
        
        # 分类部分：把卷积得到的特征图拉平后，接全连接层输出类别。
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # 32 个通道 * 7 * 7 的空间尺寸，展开成一个向量。
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # 先提取图像特征，再送进分类头。
        x = self.features(x)
        x = self.classifier(x)
        return x

# 用假的 28x28 灰度图测试网络，检查输出维度是否正确。
cnn = SimpleCNN(num_classes=10).to(device)
dummy_images = torch.randn(4, 1, 28, 28).to(device)
dummy_logits = cnn(dummy_images)
print(dummy_logits.shape)


---

## 9. 常用函数速查和易错点

### 9.1 常用函数速查

| 类别 | 函数 / 写法 | 作用 |
| --- | --- | --- |
| 随机种子 | `torch.manual_seed(42)` | 固定 CPU 随机性 |
| 设备 | `torch.device(...)` | 选择 CPU/GPU |
| 移动设备 | `model.to(device)`, `x.to(device)` | 模型和数据放到同一设备 |
| 模型参数 | `model.parameters()` | 传给优化器 |
| 参数字典 | `model.state_dict()` | 保存/加载权重 |
| 训练模式 | `model.train()` | 启用训练行为 |
| 推理模式 | `model.eval()` | 启用推理行为 |
| 关闭梯度 | `torch.no_grad()` | 验证和推理时使用 |
| 清空梯度 | `optimizer.zero_grad()` | 每个 batch 反向传播前调用 |
| 反向传播 | `loss.backward()` | 计算梯度 |
| 更新参数 | `optimizer.step()` | 根据梯度更新模型 |
| 预测类别 | `logits.argmax(dim=1)` | 取最大 logit 对应类别 |
| 预测概率 | `torch.softmax(logits, dim=1)` | 展示分类概率 |

### 9.2 常见易错点

| 问题 | 表现 | 解决方式 |
| --- | --- | --- |
| 忘记清空梯度 | loss 变化异常，梯度累加 | 每个 batch 前调用 `optimizer.zero_grad()` |
| 训练前手动 softmax | `CrossEntropyLoss` 效果异常 | 直接把 logits 传给 `CrossEntropyLoss` |
| 标签类型错误 | 报 dtype 相关错误 | 多分类标签用 `y.long()` |
| 模型和数据不在同一设备 | 报 CPU/CUDA device mismatch | 同时使用 `.to(device)` |
| 忘记 `model.eval()` | 验证结果不稳定 | 验证/推理前调用 `model.eval()` |
| 验证时没关梯度 | 显存占用变大 | 用 `with torch.no_grad()` |
| batch 维度缺失 | 形状不匹配 | 单张样本用 `x.unsqueeze(0)` 补 batch 维度 |
| 学习率过大 | loss 不下降或震荡 | 降低 `lr`，如从 `1e-3` 降到 `1e-4` |
| 学习率过小 | loss 降得很慢 | 适当增大 `lr` 或使用 scheduler |

### 9.3 一个最小训练模板

```python
model = MyModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
```

***********